In [5]:
import sys
sys.path.append('../src')

import torch
import numpy as np
import pandas as pd
from pathlib import Path

# Import models
from models.hybrid_model import HybridChagasModel

# Import training
from training.dataset import create_dataloaders
from training.trainer import ChagasTrainer

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

# Device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

ModuleNotFoundError: No module named 'models'

In [6]:
# Fold number - CHANGE THIS FOR EACH NOTEBOOK
FOLD = 0  # Change to 1, 2, 3, 4 for other notebooks

# Paths
DATA_DIR = Path('../../data/processed')
METADATA_CSV = DATA_DIR / 'metadata/combined_5fold.csv'
IMAGES_DIR = DATA_DIR / '2d_images'
SIGNALS_DIR = DATA_DIR / '1d_signals_100hz'

# Checkpoints
CHECKPOINT_DIR = Path('../../checkpoints')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Pretrained weights
MAE_CHECKPOINT = '../../checkpoints/mae_2d_pretrained.pt'
STMEM_CHECKPOINT = '../../checkpoints/stmem_1d_pretrained.pt'

# Training config
BATCH_SIZE = 32
NUM_WORKERS = 4
USE_AMP = True  # Mixed precision

# Phase 1 (FM frozen)
PHASE1_ITERATIONS = 2000
PHASE1_LR = 2e-4

# Phase 2 (FM unfrozen)
PHASE2_ITERATIONS = 12000
PHASE2_LR_HIGH = 2e-4  # For alignment + classifier
PHASE2_LR_LOW = 2e-5   # For 2D-ViT + FM (10× lower)

print(f"✓ Configuration set for Fold {FOLD}")

✓ Configuration set for Fold 0


In [7]:
train_loader, val_loader = create_dataloaders(
    metadata_csv=str(METADATA_CSV),
    images_dir=str(IMAGES_DIR),
    signals_dir=str(SIGNALS_DIR),
    fold=FOLD,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    use_weighted_sampling=True,  # 5× oversample positives
    augment_train=True
)

# Test batch
batch = next(iter(train_loader))
print(f"\n✓ Batch shapes:")
print(f"  image: {batch['image'].shape}")
print(f"  signal: {batch['signal'].shape}")
print(f"  age: {batch['age'].shape}")
print(f"  sex: {batch['sex'].shape}")
print(f"  label: {batch['label'].shape}")

NameError: name 'create_dataloaders' is not defined

In [8]:
model = HybridChagasModel(
    img_size=(24, 2048),
    patch_size_2d=(8, 64),  # CORRECTED: divides evenly
    num_leads=12,
    seq_len_1d=1000,
    patch_size_1d=50,
    embed_dim=768,
    depth=12,
    num_heads=12,
    use_aol=True,
    use_demographics=True
)

# Load pretrained weights
if Path(MAE_CHECKPOINT).exists():
    print(f"\n✓ Loading MAE pretrained weights...")
    model.vit_2d.load_mae_pretrained(MAE_CHECKPOINT)
else:
    print(f"\n⚠️  MAE checkpoint not found: {MAE_CHECKPOINT}")
    print("   Training 2D-ViT from scratch...")

if Path(STMEM_CHECKPOINT).exists():
    print(f"\n✓ Loading ST-MEM pretrained weights...")
    model.vit_1d_fm.load_stmem_pretrained(STMEM_CHECKPOINT)
else:
    print(f"\n⚠️  ST-MEM checkpoint not found: {STMEM_CHECKPOINT}")
    print("   Training 1D-ViT FM from scratch...")

model = model.to(device)
print(f"\n✓ Model created and moved to {device}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")

NameError: name 'HybridChagasModel' is not defined

In [9]:
trainer = ChagasTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    phase1_iterations=PHASE1_ITERATIONS,
    phase2_iterations=PHASE2_ITERATIONS,
    phase1_lr=PHASE1_LR,
    phase2_lr_high=PHASE2_LR_HIGH,
    phase2_lr_low=PHASE2_LR_LOW,
    checkpoint_dir=str(CHECKPOINT_DIR),
    use_amp=USE_AMP
)

print("✓ Trainer created")

NameError: name 'ChagasTrainer' is not defined

In [10]:
# Train!
metrics = trainer.train(fold=FOLD)

print(f"\n{'='*60}")
print(f"Final Results - Fold {FOLD}")
print(f"{'='*60}")
print(f"AUROC:     {metrics['auroc']:.4f}")
print(f"AUPRC:     {metrics['auprc']:.4f}")
print(f"TPR@5%:    {metrics['tpr_5pct']:.4f} ⭐ PRIMARY METRIC")
print(f"{'='*60}")

# Target check
if metrics['tpr_5pct'] >= 0.42:
    print("✅ TARGET ACHIEVED! (≥0.42)")
elif metrics['tpr_5pct'] >= 0.35:
    print("⚠️  Good progress, but below target 0.42")
else:
    print("❌ Below minimum threshold 0.35")

NameError: name 'trainer' is not defined

In [ ]:
# Save metrics to CSV
results_df = pd.DataFrame([metrics])
results_df['fold'] = FOLD
results_csv = CHECKPOINT_DIR / f"fold{FOLD}_results.csv"
results_df.to_csv(results_csv, index=False)

print(f"✓ Results saved to {results_csv}")